<a href="https://colab.research.google.com/github/BBVA/mercury-graph/blob/master/tutorials/mercury-graph-tutorial-fifa-nospark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Table of contents
- [What is `mercury-graph`?](#mercury-graph)
- [Environment setup](#environment-setup)
- [Graph creation](#graph-creation)
- [Accessing graph properties](#graph-properties)
- [Graph embeddings](#graph-embeddings)
- [Spectral clustering](#spectral)
- [Transition matrix (Markov chains)](#transition)

# What is `mercury-graph`? <a name="mercury-graph"></a>

**`mercury-graph`** is a Python library that offers **graph analytics capabilities with a technology-agnostic API**, enabling users to apply a curated range of performant and scalable algorithms and utilities regardless of the underlying data framework. The consistent, scikit-like interface abstracts away the complexities of internal transformations, allowing users to effortlessly switch between different graph representations to leverage optimized algorithms implemented using pure Python, [**numba**](https://numba.pydata.org/), [**networkx**](https://networkx.org/) and PySpark [**GraphFrames**](https://graphframes.github.io/graphframes/docs/_site/index.html).

It is a part of [**`mercury`**](https://www.bbvaaifactory.com/mercury/), a collaborative library developed by the **Advanced Analytics community at BBVA** that offers a broad range of tools to simplify and accelerate data science workflows. This library was originally an Inner Source project, but some components, like `mercury.graph`, have been released as Open Source.

Currently implemented **submodules** in `mercury.graph` include:
- [**`mercury.graph.core`**](#graph-creation), with the main classes of the library that create and store the graphs' data and properties.
- **`mercury.graph.ml`**, with graph theory and machine learning algorithms such as [Louvain community detection](#louvain), [spectral clustering](#spectral), [Markov chains](#transition), [spreading activation-based diffusion models](#spread-activation) and graph random walkers.
- **`mercury.graph.embeddings`**, with classes that calculate [graph embeddings](#graph-embeddings) in different ways, such as following the [Node2Vec](#node2vec) algorithm.


# Environment setup <a name="environment-setup"></a>



<div class="alert alert-block alert-info">
<b>Note:</b> This notebook will showcase methods in `mercury.graph` that do not require configuration of a Spark cluster.
</div>

In [ ]:
# Mercury-Graph
! pip install mercury-graph

From `mercury.graph`, we first **import `Graph`**, to create graphs **from pandas/Spark dataframes or from [networkx](https://networkx.org/)/[graphframes](https://graphframes.github.io/graphframes/docs/_site/index.html) graph objects**. It is the core class of the library, storing the graphs' data and properties and offering a **flexible and technology-agnostic API**.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)

import matplotlib.pyplot as plt
import seaborn as sns

import networkx as nx

from mercury.graph.core import Graph

# Graph creation <a name="graph-creation"></a>

We will create a graph `g` based on the [**FIFA 2017 dataset**](https://www.kaggle.com/datasets/artimous/complete-fifa-2017-player-dataset-global):
- From the original dataset, since players lack a proper ID, duplicated names were removed.
- Edges are created by random sampling the interactions between any two players.
- Players who share the same club or the same position count 1, if both then 2 (column `times`, which will represent the weights of the edges of the graph).

In [ ]:
df_edges = pd.read_csv("https://raw.githubusercontent.com/BBVA/mercury-graph/refs/heads/master/tutorials/data/fifa17_edges.csv", sep="\t")

df_edges[(df_edges["ori"]=="Gianluigi Buffon") |
         (df_edges["dest"]=="Gianluigi Buffon")].head()

From the original node (player) attributes, just the club and the position are kept.

In [ ]:
df_nodes = pd.read_csv("https://raw.githubusercontent.com/BBVA/mercury-graph/refs/heads/master/tutorials/data/fifa17_nodes.csv", sep="\t")

df_nodes[df_nodes["name"]=="Gianluigi Buffon"]

In [ ]:
g = Graph(data=df_edges,
          nodes=df_nodes,
          keys={"src": "ori",
                "dst": "dest",
                "weight": "times",
                "id": "name"})

print(g)

Given that `Graph` has been provided pandas dataframes as inputs, a [networkx](https://networkx.org/) object has been created internally (`has_networkx: True`). If Spark dataframes are provided, a [graphframes](https://graphframes.github.io/graphframes/docs/_site/index.html) object is created internally (`has_graphframe: True`).

To execute some algorithms faster and better visualize the results, two **subgraphs** will be created:
- `g_italy`, containing all players that are part of **Italian clubs**.
- `g_roma_11`, containing only the subset of **players from AS Roma's starting eleven** as nodes.
Subgraphs can be built filtering node attributes using networkx methods.

In [ ]:
italy_clubs = ["Atalanta", "Bologna", "Cagliari", "Chievo Verona", "Crotone", "Empoli", "Fiorentina", "Genoa", "Inter", "Juventus", "Lazio", "Milan", "Napoli", "Palermo", "Pescara", "Roma", "Sampdoria", "Sassuolo", "Torino", "Udinese"]
italy_players = [node for node, attrs in g.networkx.nodes(data=True)
                 if attrs["club"] in italy_clubs]

g_italy = Graph(g.networkx.subgraph(italy_players))
print(g_italy)

In [ ]:
starting_roma_players = [node for node, attrs in g.networkx.nodes(data=True)
                         if attrs["club"]=="Roma"
                         and not attrs["position"] in ["Res", "Sub"]]

g_roma_11 = Graph(g.networkx.subgraph(starting_roma_players))
print(g_roma_11)

Next, some functions and utilities are defined for plotting graphs and results.

In [ ]:
def plot_football_field(g, player_positions, node_colors):
  plt.figure(figsize=(10, 8))
  plt.gca().set_facecolor(color_palette["whiteblue"])
  nx.draw_networkx(g.networkx,
                   pos=player_positions,
                   node_color=node_colors,
                   font_size=6,
                   font_weight="bold",
                   font_color="black",
                   edge_color=color_palette["darkblue"])

color_palette = {
    'navy': '#072146',
    'darkblue': '#004481',
    'darkaqua': '#028484',
    'blue': '#1973b8',
    'aqua': '#2dcccd',
    'lightblue': '#49a5e6',
    'whiteblue': '#d4edfc',
}

default_node_colors = lambda node_data: [color_palette["aqua"] if attr["position"]=="GK" else
                                         color_palette["darkaqua"] if attr["position"] in ["LB", "RB", "LCB", "CB", "RCB"] else
                                         color_palette["blue"] if attr["position"] in ["RM", "LM", "RDM", "LDM", "RCM", "LCM", "CAM"] else
                                         color_palette["lightblue"] for n, attr in node_data]

In [ ]:
player_positions_roma = {
    "Wojciech Szczsny": (0.5, 0.5),
    "Antonio Rudiger": (-0.25, 1.4),
    "Federico Fazio": (0.5, 1.15),
    "Kostas Manolas": (1.25, 1.4),
    "Emerson": (-0.8, 2.5),
    "Kevin Strootman": (0, 2.10),
    "Daniele De Rossi": (1, 2.10),
    "Bruno Peres": (1.8, 2.5),
    "Radja Nainggolan": (-0.3, 3.1),
    "Edin Deko": (0.5, 2.7),
    "Mohamed Salah": (1.3, 3.1),
}

plot_football_field(g_roma_11,
                    player_positions=player_positions_roma,
                    node_colors=default_node_colors(g_roma_11.networkx.nodes(data=True)))

# Accessing graph properties <a name="graph-properties"></a>

Given that `Graph` has been provided pandas dataframes as inputs to create object `g`, a networkx object has been created internally (`has_networkx: True`). However, at any point, the networkx or graphframes representation of the `Graph` object can be accessed through properties `g.networkx` and `g.graphframe`, respectively. The objects are created when accessing these properties for the first time and are stored for later use. All classes and methods in `mercury.graph` that expect a `Graph` object as input internally perform the required conversions (if necessary) according to each algorithm's implementation.

For example, we can access the **networkx representation** of the created graph to use networkx's methods and API:

In [ ]:
print(f"Club: {g.networkx.nodes(data='club')['Gianluigi Buffon']}")
print(f"Position: {g.networkx.nodes(data='position')['Gianluigi Buffon']}")

Several other **properties** can be accessed from the `Graph` object, **regardless of the underlying technology**:
- PageRank algorithm
- Degree, indegree, outdegree of the nodes
- Connected components
- Betweenness centrality, closeness centrality

The **PageRank algorithm** (see paper [here](https://www.semanticscholar.org/paper/The-PageRank-Citation-Ranking-%3A-Bringing-Order-to-Page-Brin/eb82d3035849cd23578096462ba419b53198a556)) assigns a numerical score to each node in the graph, representing its relative importance within the graph. A higher score means the node is more important or influential (these nodes are typically well-connected and linked to other high-scoring nodes). In this implementation, scores are normalized to sum up to 1.

In [ ]:
pagerank_result = g_roma_11.pagerank

print(f"Total sum: {sum(pagerank_result.values())}\n")

# Get top 10 results from the pagerank dictionary
dict(sorted(pagerank_result.items(), key=lambda item: item[1], reverse=True)[:10])

In [ ]:
name = "Daniele De Rossi"
print(f"Degree: {g_roma_11.degree[name]}")
print(f"Indegree: {g_roma_11.in_degree[name]}")
print(f"Outdegree: {g_roma_11.out_degree[name]}")
print(f"Connected components: {g_roma_11.connected_components[name]}")
print(f"Betweenness centrality: {g_roma_11.betweenness_centrality[name]}")
print(f"Closeness centrality: {g_roma_11.closeness_centrality[name]}")

In [ ]:
print(f"Number of connected components: {len(set([v['cc_id'] for k, v in g_roma_11.connected_components.items()]))}\n")

list(g_roma_11.connected_components.items())

As could be seen in the plot, all nodes of the AS Roma subgraph are connected and thus there is only one connected component. If we create another subgraph with the interactions between the players of the starting eleven of Juventus FC, three connected components can be observed and are indeed detected.

In [ ]:
starting_juve_players = [node for node, attrs in g.networkx.nodes(data=True)
                         if attrs["club"]=="Juventus"
                         and not attrs["position"] in ["Res", "Sub"]]
g_juve_11 = Graph(g.networkx.subgraph(starting_juve_players))

player_positions_juve = {
      "Gianluigi Buffon": (0.5, 0.5),
      "Stephan Lichtsteiner": (-0.7, 1.3),
      "Leonardo Bonucci": (1, 1),
      "Giorgio Chiellini": (0, 1),
      "Alex Sandro": (1.7, 1.3),
      "Juan Cuadrado": (-0.8, 2.5),
      "Sami Khedira": (0, 2),
      "Miralem Pjanic": (1, 2),
      "Mario Mandukic": (1.8, 2.5),
      "Paulo Dybala": (0.5, 2.8),
      "Gonzalo Higuain": (0.5, 3.6)
  }

plot_football_field(g_juve_11, player_positions=player_positions_juve, node_colors=default_node_colors(g_juve_11.networkx.nodes(data=True)))

In [ ]:
print(f"Number of connected components: {len(set([v['cc_id'] for k, v in g_juve_11.connected_components.items()]))}\n")

list(g_juve_11.connected_components.items())

# Graph embeddings <a name="graph-embeddings"></a>

Class `GraphEmbeddings` of mercury.graph.embeddings creates an **embedding mapping the nodes of a graph by doing random walks**, implemented using numpy and networkx. These walks start from a random node and select the edges with a probability that is proportional to the **weight** of the edge.

In [ ]:
from mercury.graph.embeddings import GraphEmbedding

In [ ]:
ge = GraphEmbedding(dimension=140,
                    n_jumps=100000,
                    max_per_epoch=1000,
                    learn_step=1,
                    bidirectional=True)

print(ge)

In [ ]:
ge.fit(g_italy)

After fitting the object to the subgraph, an `Embedding` object is created, containing the representation of the vector embeddings matrix.

In [ ]:
print(ge.embedding(), "\n")

ge_em_np = ge.embedding().as_numpy()
print(f"Shape: {ge_em_np.shape} \n")
print(ge_em_np)

For each node, the **most similar nodes and the similarity metric** (by default, cosine similarity) can be obtained using method `get_most_similar_nodes`. Essentially, it fetches the most similar embeddings using the underlying `Embedding` object.

In [ ]:
similarity_df_ge = ge.get_most_similar_nodes("Gianluigi Buffon", 5)
similarity_df_ge

Many of the most similar players to Gianluigi Buffon are also Juventus players, given how the weights and edges have been computed in the dataset.

In [ ]:
df_nodes[df_nodes["name"].isin(similarity_df_ge["word"].tolist())]

The embeddings can be visualized to verify that **players in the same club are usually close to each other in the embedding space** by first reducing the embedding dimension to 2 using [**TSNE**](https://scikit-learn.org/1.5/modules/generated/sklearn.manifold.TSNE.html).

Each color in the plot represents a club, and the clusters are generally distinguishable. Note that not all players within a club are connected with each other and that two players with the same position can also be connected even if they belong to different clubs - hence the irregular clusters.

In [ ]:
from sklearn.manifold import TSNE

# Condense embeddings per node into a list
ge_em_df = pd.DataFrame({"vector": [v for v in ge_em_np]})
# Add player names (node IDs, not present in the embedding matrix)
ge_em_df["name"] = list(g_italy.networkx.nodes)
# Add club (node property)
ge_em_df = ge_em_df.merge(df_nodes[["name", "club"]], on="name")

# Apply TSNE
ge_tsne = TSNE(perplexity=12.0, metric='euclidean', random_state=1)
ge_tsne_np = ge_tsne.fit_transform(np.stack(ge_em_df["vector"].values))

# Visualize results
ge_tsne_pd = pd.DataFrame(ge_tsne_np)
ge_tsne_pd["club"] = ge_em_df["club"].values

fig, axes = plt.subplots(1, 1, figsize=(12, 6))
ax = sns.scatterplot(x=0, y=1,
                     hue="club",
                     palette="tab20",
                     data=ge_tsne_pd)
ax.legend_.remove()
plt.show()

# Spectral clustering <a name="spectral"></a>

Class `SpectralClustering` from `mercury.graph.ml` implements the **unsupervised [spectral clustering algorithm](https://www.sciencedirect.com/topics/computer-science/spectral-clustering)** to group nodes in a graph. This algorithm can work in two modes: "networkx" (running the algorithm locally, with a methodology similar to [scikit-learn's implementation](https://scikit-learn.org/1.5/modules/generated/sklearn.cluster.SpectralClustering.html) but expecting a graph object instead of a numpy array) or "spark" (using PySpark and graphframes).

In [ ]:
from mercury.graph.ml import SpectralClustering

In [ ]:
sc = SpectralClustering(n_clusters=3, mode="networkx")

print(sc)

As with `LouvainCommunities`, cluster assignments are available after fitting through the pandas dataframe `.labels_`, following the `scikit-learn` convention:

In [ ]:
sc.fit(g_roma_11)

In [ ]:
sc.labels_

In [ ]:
sc.labels_.groupby("cluster").agg({"node_id": lambda x: set(x)})

Again, the assignment of each node/player to a cluster can also be visualized for better interpretation. When setting the number of clusters to `k=3`, slightly different clusters are obtained compared to the partitions from `LouvainCommunities`, but the results are similar nonetheless.

In [ ]:
color_map = {
    0: "pink",
    1: "yellow",
    2: "cyan",
}
node_colors = [color_map[comm_id] for comm_id in sc.labels_["cluster"]]

plot_football_field(g_roma_11, player_positions=player_positions_roma, node_colors=node_colors)

**Modularity** is a metric that measures the strength of the division of a graph into clusters. The evolution of modularity for each number of clusters (`k`) can be observed, showing that a number of clusters between 2 and 4 should be used for best results.

In [ ]:
print(f"Modularity: {sc.modularity_}")

In [ ]:
modularities = []
num_clusters = range(1, 12)

for k in num_clusters:
  sc = SpectralClustering(n_clusters=k, mode="networkx")
  sc.fit(g_roma_11)
  modularities.append(sc.modularity_)

sns.lineplot(x=num_clusters, y=modularities)
plt.xlabel("Number of clusters (k)")
plt.xticks(ticks=num_clusters)
plt.ylabel("Modularity")
plt.show()

Spectral clustering can also be performed using PySpark under the hood by simply passing `mode="spark"` to the constructor, which uses the `Graph` object's graphframe property.

# Transition matrix (Markov chains) <a name="transition"></a>

Class `Transition` of `mercury.graph.ml` can be used to obtain the **transition matrix** of a graph, which computes the distribution of probability of being in each of the nodes (or states) of a directed graph (or Markov process).

In [ ]:
from mercury.graph.ml import Transition

In [ ]:
tm = Transition().fit(g_roma_11).to_pandas()

tm

In [ ]:
sns.heatmap(tm, cmap="Blues")
plt.show()